# NeuroScan — Stage 1: 8-class Tumor Classifier (V5)

Trains the deployed `tumor_classifier.pth` (an 8-class EfficientNet-B0) by combining three Kaggle datasets:

| Source | Slug | Has split? |
|---|---|---|
| `data1` | `masoudnickparvar/brain-tumor-mri-dataset` | yes (Training/Testing) |
| `data2` | `briscdataset/brisc2025` (classification task) | yes (train/test) |
| `data44` | `fernando2rad/brain-tumor-mri-images-44c` | no — split 80/20 here |

**Output:** `tumor_classifier_v5.pth` → rename to `tumor_classifier.pth` and place in `backend/`.

**Class index order (alphabetical, from `ImageFolder`):**
`[carcinoma, glioma, meningioma, neurocytoma, no_tumor, papilloma, pituitary, schwannoma]`
This MUST match `CLASS_NAMES` in `backend/model.py`.

Run on a Colab GPU runtime. Set the `KAGGLE_API_TOKEN` Colab Secret before running.

**Known methodological limitations** (see `ARCHITECTURE.md` for full discussion):
- Patient-level data leakage from random image-level 80/20 split on data44
- The 4 classes that exist only in data44 (schwannoma, neurocytoma, carcinoma, papilloma) carry a source confound
- Severe class imbalance (pituitary is data1-only)
- Test set doubles as validation set
- No best-checkpoint saving, no torch/numpy random seed

## 1. Setup & Download Datasets

In [ ]:
!pip install -q kaggle

In [ ]:
import os
from google.colab import userdata

# Kaggle token from Colab Secrets (left sidebar key icon).
# Add a secret named KAGGLE_API_TOKEN before running this cell.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d data1

!kaggle datasets download -d briscdataset/brisc2025
!unzip -q brisc2025.zip -d data2

!kaggle datasets download -d fernando2rad/brain-tumor-mri-images-44c
!unzip -q brain-tumor-mri-images-44c.zip -d data44

print("All datasets downloaded")

## 2. Build Expanded 8-class Dataset

Merges all three sources into an `expanded/{train,test}/<class>/` ImageFolder tree.

For data1 and data2 the existing train/test split is preserved. For data44 (which has no split) each class's image list is shuffled with `random.seed(42)` and split 80/20 — this is where image-level leakage lives.

In [ ]:
import os, shutil, random

EXPANDED_CLASSES = {
    'glioma': {
        'data1': {'Training': ['glioma'], 'Testing': ['glioma']},
        'data2': {'train': ['glioma'], 'test': ['glioma']},
        'data44': ['Astrocitoma T1', 'Astrocitoma T2', 'Astrocitoma T1C+',
                   'Glioblastoma T1', 'Glioblastoma T2', 'Glioblastoma T1C+',
                   'Oligodendroglioma T1', 'Oligodendroglioma T2', 'Oligodendroglioma T1C+',
                   'Ependimoma T1', 'Ependimoma T2', 'Ependimoma T1C+',
                   'Ganglioglioma T1', 'Ganglioglioma T2', 'Ganglioglioma T1C+'],
    },
    'meningioma': {
        'data1': {'Training': ['meningioma'], 'Testing': ['meningioma']},
        'data2': {'train': ['meningioma'], 'test': ['meningioma']},
        'data44': ['Meningioma T1', 'Meningioma T2', 'Meningioma T1C+'],
    },
    'no_tumor': {
        'data1': {'Training': ['notumor'], 'Testing': ['notumor']},
        'data2': {'train': ['no_tumor'], 'test': ['no_tumor']},
        'data44': ['_NORMAL T1', '_NORMAL T2'],
    },
    'pituitary': {
        'data1': {'Training': ['pituitary'], 'Testing': ['pituitary']},
    },
    'schwannoma': {
        'data44': ['Schwannoma T1', 'Schwannoma T2', 'Schwannoma T1C+'],
    },
    'neurocytoma': {
        'data44': ['Neurocitoma T1', 'Neurocitoma T2', 'Neurocitoma T1C+'],
    },
    'carcinoma': {
        'data44': ['Carcinoma T1', 'Carcinoma T2', 'Carcinoma T1C+'],
    },
    'papilloma': {
        'data44': ['Papiloma T1', 'Papiloma T2', 'Papiloma T1C+'],
    },
}

# Clean slate
shutil.rmtree('expanded', ignore_errors=True)
for split in ['train', 'test']:
    for cls in EXPANDED_CLASSES:
        os.makedirs(f'expanded/{split}/{cls}', exist_ok=True)

idx = 0

for cls, sources in EXPANDED_CLASSES.items():
    # data1 (has Training/Testing split)
    if 'data1' in sources:
        for split_src, folders in sources['data1'].items():
            split_dst = 'train' if split_src == 'Training' else 'test'
            for folder in folders:
                src_dir = f'data1/{split_src}/{folder}'
                if os.path.exists(src_dir):
                    for f in os.listdir(src_dir):
                        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                            shutil.copy2(f'{src_dir}/{f}', f'expanded/{split_dst}/{cls}/d1_{idx}_{f}')
                            idx += 1

    # data2/BRISC (has train/test split)
    if 'data2' in sources:
        for split_src, folders in sources['data2'].items():
            for folder in folders:
                src_dir = f'data2/brisc2025/classification_task/{split_src}/{folder}'
                if os.path.exists(src_dir):
                    for f in os.listdir(src_dir):
                        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                            shutil.copy2(f'{src_dir}/{f}', f'expanded/{split_src}/{cls}/d2_{idx}_{f}')
                            idx += 1

    # data44 (no split — random 80/20 at image level)
    if 'data44' in sources:
        imgs = []
        for folder in sources['data44']:
            src_dir = f'data44/{folder}'
            if os.path.exists(src_dir):
                for f in os.listdir(src_dir):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        imgs.append(f'{src_dir}/{f}')
        random.seed(42)
        random.shuffle(imgs)
        split_idx = int(len(imgs) * 0.8)
        for i, img_path in enumerate(imgs):
            split = 'train' if i < split_idx else 'test'
            shutil.copy2(img_path, f'expanded/{split}/{cls}/d3_{idx}_{os.path.basename(img_path)}')
            idx += 1

# Print stats
print("Expanded dataset (8 classes):\n")
total_train = total_test = 0
for split in ['train', 'test']:
    print(f"  {split}:")
    for cls in sorted(os.listdir(f'expanded/{split}')):
        n = len(os.listdir(f'expanded/{split}/{cls}'))
        print(f"    {cls:>15s}: {n}")
        if split == 'train': total_train += n
        else: total_test += n
print(f"\n  Total train: {total_train}, Total test: {total_test}")

## 3. Train V5 — EfficientNet-B0, fully unfrozen, AdamW + cosine, 20 epochs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")  # MUST say cuda

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('expanded/train', transform=train_transforms)
test_dataset = datasets.ImageFolder('expanded/test', transform=test_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Classes: {train_dataset.classes}")
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(1280, 8))
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    model.train()
    running_loss = correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    test_acc = 100.0 * correct / total
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f"V5 {epoch+1}/20 | Loss: {train_loss:.4f} | Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | LR: {lr:.6f}")

torch.save(model.cpu().state_dict(), 'tumor_classifier_v5.pth')
from google.colab import files
files.download('tumor_classifier_v5.pth')

## 4. (Optional) Collect sample MRIs for the frontend

⚠️ Caveat from the audit: this grabs `imgs[-3:]` (last filesystem-order images) from each data44 folder. The training split was made via `random.shuffle` after `random.seed(42)`, so filesystem-tail images have NO known relationship to the held-out 20% — they may have been in training. If you care about "unseen" samples, build them from a separate dataset (e.g. `sartajbhuvaji/brain-tumor-classification-mri`).

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('new_samples', exist_ok=True)
for folder, name in [
    ('Schwannoma T1C+', 'schwannoma'),
    ('Neurocitoma T1C+', 'neurocytoma'),
    ('Carcinoma T1C+', 'carcinoma'),
    ('Papiloma T1C+', 'papilloma'),
]:
    img = os.listdir(f'data44/{folder}')[0]
    shutil.copy2(f'data44/{folder}/{img}', f'new_samples/{name}.jpg')

shutil.make_archive('new_samples', 'zip', '.', 'new_samples')
files.download('new_samples.zip')